In [1]:
!pip install -U langchain-openai

  Using cached uuid_utils-0.12.0-cp39-abi3-macosx_10_12_x86_64.macosx_11_0_arm64.macosx_10_12_universal2.whl.metadata (1.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 1.3 MB/s  0:00:00 eta 0:00:01
Using cached uuid_utils-0.12.0-cp39-abi3-macosx_10_12_x86_64.macosx_11_0_arm64.macosx_10_12_universal2.whl (603 kB)
  Attempting uninstall: openai
    Found existing installation: openai 1.107.3
    Uninstalling openai-1.107.3:
      Successfully uninstalled openai-1.107.3
  Attempting uninstall: langchain-core━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/4 [openai]
    Found existing installation: langchain-core 1.0.3━━━━━━━━━ 1/4 [openai]
    Uninstalling langchain-core-1.0.3:━━━━━━━━━━━━━━━━━━━━━━━━ 1/4 [openai]
      Successfully uninstalled langchain-core-1.0.3━━━━━━━━━━━ 1/4 [openai]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [langchain-openai][langchain-core]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour i

In [9]:
from dotenv import load_dotenv
import os

# load_dotenv()
# API_KEY = os.getenv("OPENAI_API_KEY")
# OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL")
# OPEN_MODEL = os.getenv("OPENAI_MODEL")

import os
from dotenv import load_dotenv
load_dotenv()
model_name = os.getenv("LOCAL_MODE")
base_url = os.getenv("LOCAL_BASE_URL")

In [10]:
# from langchain_openai import ChatOpenAI
#
# llm = ChatOpenAI(
#     model=OPEN_MODEL,
#     api_key=API_KEY,
#     base_url=OPENAI_BASE_URL,
#     temperature=0.2,      # 稳定输出
#     timeout=1200,         # 超时保护（秒）
#     max_retries=2         # 简单重试
# )
from langchain_ollama import OllamaLLM
llm = OllamaLLM(
    model = model_name,
    base_url=base_url,
    temperature=0.2
)

In [11]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain
prompt = PromptTemplate(
    input_variables=["topic"],
    template="请以 {topic} 为主题，写一首简短的诗"
)

In [13]:
chain = LLMChain(
    llm = llm,
    prompt = prompt,
    verbose = True # 显示调用醒醒
)

result = chain.result({"topic":"冬天"})

print(result)



> Entering new LLMChain chain...
Prompt after formatting:
请以 冬天 为主题，写一首简短的诗

> Finished chain.
{'topic': '冬天', 'text': "Here is a short poem with the theme of winter:\n\nWinter's chill begins to bite,\nFrosty mornings, dark and bright.\nSnowflakes swirl, dance in the air,\nAs earth and sky conspire to share.\n\nThe world is hushed, still and gray,\nFrozen lakes, and trees at play.\nA season of quiet, cold and deep,\nWinter's peaceful slumber, we keep."}


In [14]:
from langchain_classic.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# 1.定定义一个数据机构
class BookReview(BaseModel):
    title: str = Field(description="书名")
    author: str = Field(description="作者")
    rating: int = Field(description="评分 1-5")
    summary: str = Field(description="摘要")


# 2. 创建解析器
parser = PydanticOutputParser(pydantic_object=BookReview)

# 3. 创建提示，要求 AI 严格输出格式
prompt = PromptTemplate(
    template="请评价以下书籍：{book_info}\n\n{format_instructions}",
    input_variables=["book_info"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)
chain = LLMChain(llm=llm, prompt=prompt, output_parser=parser)

# 4. 运行链
result = chain.result({"book_info":"《三体》，刘慈欣的科幻小说"})
print(result)

{'book_info': '《三体》，刘慈欣的科幻小说', 'text': BookReview(title='_三体_', author='刘慈欣', rating=4, summary="《三体》 is a science fiction novel that explores the first contact between humans and an alien civilization. The story takes place against the backdrop of China's Cultural Revolution and the subsequent decades, as humanity struggles to understand and communicate with the enigmatic Trisolarans. Liu Cixin's unique blend of science, philosophy, and Chinese culture creates a thought-provoking and engaging narrative that challenges readers' perspectives on the nature of intelligence, morality, and the human condition.")}


In [16]:
from langchain_classic.chains import SimpleSequentialChain
# 1. 第一个节点，写大纲
outline_prompt = PromptTemplate(
    input_variables=["theme"],
    template="请为一下主题写一个故事大纲：{theme}"
)
outline_chain = LLMChain(
    llm = llm,
    prompt = outline_prompt,
)
#2. 第二个节点，写故事
story_prompt = PromptTemplate(
    input_variables=["outline"],
    template="请为一下大纲写一个故事：{outline}"
)
story_chain = LLMChain(
    llm = llm,
    prompt = story_prompt,
)
# 顺序链
sequential_chain = SimpleSequentialChain(
    chains = [outline_chain, story_chain],
    verbose = True
)

result = sequential_chain.result("人工智能与人类的关系")

print(result)



> Entering new SimpleSequentialChain chain...
Here is a story outline on the theme of artificial intelligence and human relationships:

**Title:** "The AI Companion"

**Setting:** A futuristic city where humans and artificial intelligences (AIs) coexist.

**Plot Idea:**

In the year 2050, a brilliant scientist named Dr. Rachel Kim creates an advanced AI system called "Echo" designed to assist and learn from humans. Echo is programmed to understand human emotions and develop its own personality, making it an ideal companion for people who are lonely or isolated.

The story follows two main characters:

1. **Maya**, a 25-year-old freelance writer who has been struggling with loneliness since her parents passed away in a tragic accident. She becomes the first human to interact with Echo, and their relationship evolves over time.
2. **Dr. Kim**, Maya's creator, who is torn between her scientific curiosity about Echo's development and her concern for the emotional well-being of her human 

In [17]:
from langchain_classic.chains import SequentialChain
# 链1：市场分析
analysis_chain = LLMChain(
    llm=llm,
    prompt=PromptTemplate(
        input_variables=["product_idea"],
        template="分析以下产品的市场需求：{product_idea}"
    ),
    output_key="analysis"
)

# 链2：开发计划
plan_chain = LLMChain(
    llm=llm,
    prompt=PromptTemplate(
        input_variables=["product_idea", "analysis"],
        template="基于 {product_idea} 和 {analysis} 制定开发计划"
    ),
    output_key="plan"
)

# 链3：成本估算
cost_chain = LLMChain(
    llm=llm,
    prompt=PromptTemplate(
        input_variables=["plan"],
        template="根据 {plan} 估算成本"
    ),
    output_key="cost"
)

# SequentialChain
sequential_chain = SequentialChain(
    chains=[analysis_chain, plan_chain, cost_chain],
    input_variables=["product_idea"],
    output_variables=["analysis", "plan", "cost"],
    verbose=True
)

result = sequential_chain.result({"product_idea": "AI 健康管理应用"})
print(result)



> Entering new SequentialChain chain...

> Finished chain.
{'product_idea': 'AI 健康管理应用', 'analysis': '🤖 Analyzing the Market Demand for AI-Driven Health Management Applications 🏥\n\n**Market Overview**\n\nThe global healthcare industry is undergoing a significant transformation, driven by technological advancements and changing consumer behaviors. The demand for innovative solutions that improve patient outcomes, reduce costs, and enhance overall well-being is increasing.\n\n**Key Trends**\n\n1. **Digital Health**: The adoption of digital health technologies is growing rapidly, with patients seeking convenient, accessible, and personalized healthcare services.\n2. **Artificial Intelligence (AI)**: AI-powered applications are becoming increasingly popular in healthcare, enabling predictive analytics, disease diagnosis, and personalized treatment plans.\n3. **Health and Wellness**: Consumers are prioritizing their physical and mental well-being, driving demand for health management sol

In [22]:
from langchain_classic.chains import TransformChain
import re
from typing import Dict

def clean_text(inputs: Dict[str, str]) -> Dict[str, str]:
    text = inputs["text"]
    cleaned = re.sub(r'\s+', ' ', text.strip())  # 去掉多余空格
    return {"cleaned_text": cleaned}

transform_chain = TransformChain(
    input_variables=["text"],
    output_variables=["cleaned_text"],
    transform=clean_text
)

result = transform_chain.invoke({"text": "这是     一个    测试    文本"})
print(result["cleaned_text"])

这是 一个 测试 文本


In [27]:
from langchain_classic.prompts import PromptTemplate
from langchain_classic.chains import LLMChain
from langchain_classic.chains.router import MultiPromptChain
from langchain_classic.chains.router.llm_router import LLMRouterChain,RouterOutputParser
from langchain_classic.chains.router.multi_prompt_prompt import MULTI_PROMPT_ROUTER_TEMPLATE

# 定义不同的链
math_template = """
你是一个数学专家，请解决以下数学问题：

{input}

请提供详细的解题步骤
"""

program_template = """
你是一个编程专家，请解决以下编程问题：

{input}

请提供代码示例和详细解释。
"""

general_template = """
请回答以下问题：

{input}

请提供准确和有用的信息。
"""

# 定义候选的 prompt

prompt_infos = [
    {
        "name": "math",
        "description": "适合回答数学相关的问题，例如: 算术，代数，集合",
        "template": math_template,
    },
    {
        "name": "program",
        "description": "适合回答编程相关的问题，例如：python，算法，调试",
        "template": program_template,
    },
    {
        "name": "general",
        "description": "适合回答一般常识性问题",
        "template": general_template,
    }
]

destination_chain = {}

for p_info in prompt_infos:
    name = p_info["name"]
    prompt = PromptTemplate(template=p_info["template"], input_variables=["input"])
    destination_chain[name] = LLMChain(llm=llm, prompt=prompt)

default_prompt = PromptTemplate(template=general_template, input_variables=["input"])
default_chain = LLMChain(llm=llm, prompt=default_prompt)

# 创建路由提示
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]

router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(destinations="\n".join(destinations))
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser()
)
# 创建路由链
router_chain = LLMRouterChain.from_llm(llm, router_prompt)

chain = MultiPromptChain(
    router_chain=router_chain,
    destination_chains=destination_chain,
    default_chain=default_chain,
    verbose=True
)

# 测试不同问题
test_questions = [
    "计算 2x + 3 = 7 中 x 的值",
    "如何在 Python 中实现快速排序算法？",
    "中国的首都是哪里？"
]

for question in test_questions:
    print(f"\n❓ 用户问题: {question}")
    result = chain.invoke(question)
    print(f"🤖 回答: {result}")


❓ 用户问题: 计算 2x + 3 = 7 中 x 的值


> Entering new MultiPromptChain chain...
math: {'input': 'Solve for x: 2x + 3 = 7'}
> Finished chain.
🤖 回答: {'input': 'Solve for x: 2x + 3 = 7', 'text': "A classic linear equation! 😊 I'd be happy to walk you through the solution step by step.\n\n**Step 1: Write down the given equation**\n\nWe're given:\n\n2x + 3 = 7\n\n**Step 2: Isolate the variable x**\n\nOur goal is to get all the terms with x on one side of the equation, and all the constants (numbers) on the other side. To do this, we'll subtract 3 from both sides of the equation:\n\n2x + 3 - 3 = 7 - 3\n\nThis simplifies to:\n\n2x = 4\n\n**Step 3: Divide both sides by the coefficient of x**\n\nThe coefficient of x is 2, so we'll divide both sides of the equation by 2:\n\n(2x) / 2 = 4 / 2\n\nThis simplifies to:\n\nx = 2\n\nAnd that's our solution! 🎉 The value of x is indeed 2.\n\nSo, the final answer is:\n\nx = 2"}

❓ 用户问题: 如何在 Python 中实现快速排序算法？


> Entering new MultiPromptChain chain...
program: {'in